This project demonstrates the implementation of a Convolutional Neural Network (CNN) using TensorFlow and Keras to classify handwritten digits (0-9) from the classic MNIST dataset.

### 1. Importing Dependencies
First, we import the necessary libraries. We use `tensorflow.keras` to access the MNIST dataset, build our sequential CNN architecture, and perform data preprocessing.

*(Note: Cleaned up unused libraries to keep the environment lightweight).*

In [7]:
# Import numerical computation library
import numpy as np

# Import Keras modules to build the neural network architecture
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# Import dataset and preprocessing utilities
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical # Updated from np_utils for modern TF versions

### 2. Loading the MNIST Dataset
The MNIST dataset comes pre-loaded in Keras. We split it into training data (to teach the model) and testing data (to evaluate its performance). We also print the shapes of the arrays to understand our data structure.

In [8]:
# Load the MNIST dataset, which is already divided into train and test sets
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Output the dimensions of our datasets
# Expected shape: (Number of Images, Height, Width)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (60000, 28, 28)
y_train shape: (60000,)
X_test shape: (10000, 28, 28)
y_test shape: (10000,)


### 3. Image Preprocessing
Before feeding images into a CNN, we must format the data correctly:
1. **Reshaping:** TensorFlow's `Conv2D` layers expect input data in the format `(batch, height, width, channels)`. Since MNIST images are grayscale, we add a single channel dimension at the end.
2. **Normalization:** Pixel values range from 0 to 255. We cast the data to `float32` and divide by 255 to scale all values between 0.0 and 1.0. This helps the neural network converge much faster during training.

In [9]:
# Reshape arrays to include the grayscale channel dimension (1)
feature_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
feature_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

# Convert data type to float32 before normalization to maintain precision
feature_train = feature_train.astype('float32')
feature_test = feature_test.astype('float32')

# Min-Max Normalization: Scale pixel values to a range of [0, 1]
feature_train /= 255.0
feature_test /= 255.0

print(f"Reshaped and Normalized feature_train: {feature_train.shape}")
print(f"Reshaped and Normalized feature_test: {feature_test.shape}")

Reshaped and Normalized feature_train: (60000, 28, 28, 1)
Reshaped and Normalized feature_test: (10000, 28, 28, 1)


### 4. One-Hot Encoding the Labels
Our target variable contains integers from 0 to 9 representing the handwritten digits. We need to convert these into categorical (one-hot encoded) vectors so the model can output a probability distribution across all 10 possible classes.

In [10]:
# Convert the numerical labels to one-hot encoded vectors (10 classes for digits 0-9)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"One-Hot encoded y_train shape: {y_train.shape}")
print(f"One-Hot encoded y_test shape: {y_test.shape}")

One-Hot encoded y_train shape: (60000, 10)
One-Hot encoded y_test shape: (10000, 10)


### 5. Defining the Model Architecture
We build a Sequential model with the following structure:
* **Conv2D Layers:** Extract spatial features (like edges and curves) using convolutional filters.
* **MaxPooling2D Layers:** Downsample the image, reducing computational load and preventing overfitting.
* **Flatten Layer:** Converts the 2D feature maps into a 1D vector to feed into standard dense layers.
* **Dense Output Layer:** Uses a `softmax` activation function to output the final probabilities for the 10 digit classes.

In [11]:
# Initialize a Sequential model
model = Sequential()

# Add the first Convolutional layer (32 filters, 3x3 kernel) and explicitly define the input shape
model.add(Conv2D(32, (3, 3), input_shape=(28, 28, 1), activation='relu'))
# Add Max Pooling to reduce spatial dimensions
model.add(MaxPooling2D(pool_size=(2, 2)))

# Add a second Convolutional layer (64 filters, 3x3 kernel)
model.add(Conv2D(64, (3, 3), activation='relu'))
# Add a second Max Pooling layer
model.add(MaxPooling2D(pool_size=(2, 2)))

# Flatten the 2D matrices into a 1D vector
model.add(Flatten())

# Add a fully connected Hidden layer with 128 neurons
model.add(Dense(128, activation='relu'))

# Add the Output layer with 10 neurons (one for each class) and softmax activation
model.add(Dense(10, activation='softmax'))

# Compile the model with the Adam optimizer and Categorical Crossentropy loss
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the architecture and parameter count
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

### 6. Training the Network
We train the model for 10 epochs. We use a batch size of 200, meaning the model will process 200 images at a time before updating its weights.

In [13]:
# Train the model on the training data
# verbose=2: prints one log line per epoch
history = model.fit(feature_train, y_train, epochs=10, batch_size=200, verbose=2)

Epoch 1/10
300/300 - 46s - 154ms/step - accuracy: 0.9974 - loss: 0.0083
Epoch 2/10
300/300 - 78s - 259ms/step - accuracy: 0.9975 - loss: 0.0077
Epoch 3/10
300/300 - 41s - 138ms/step - accuracy: 0.9981 - loss: 0.0059
Epoch 4/10
300/300 - 83s - 276ms/step - accuracy: 0.9979 - loss: 0.0059
Epoch 5/10
300/300 - 80s - 268ms/step - accuracy: 0.9989 - loss: 0.0036
Epoch 6/10
300/300 - 42s - 139ms/step - accuracy: 0.9989 - loss: 0.0032
Epoch 7/10
300/300 - 82s - 274ms/step - accuracy: 0.9990 - loss: 0.0031
Epoch 8/10
300/300 - 41s - 136ms/step - accuracy: 0.9978 - loss: 0.0063
Epoch 9/10
300/300 - 42s - 138ms/step - accuracy: 0.9983 - loss: 0.0048
Epoch 10/10
300/300 - 84s - 279ms/step - accuracy: 0.9992 - loss: 0.0023


### 7. Evaluating Model Performance
Finally, we evaluate the trained model on our unseen test data to check its final accuracy and ensure it hasn't overfit to the training set.

In [14]:
# Evaluate the model on the testing data
test_loss, test_accuracy = model.evaluate(feature_test, y_test, verbose=1)

print(f"\nFinal Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9907 - loss: 0.0426

Final Test Loss: 0.0426
Final Test Accuracy: 99.07%
